# Bab 11
# Neural Networks That Understand Language

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Deep Learning

## 1. Tujuan Bab

Bab ini menjelaskan bagaimana jaringan saraf dapat mempelajari pola dari bahasa. Kata tidak dapat langsung diproses sebagai makna oleh komputer; teks perlu diubah menjadi representasi numerik yang dapat digunakan sebagai masukan jaringan.

Fokus pembahasan bab:
1. Natural Language Processing (NLP) dan tugas-tugasnya.
2. Klasifikasi sentimen ulasan film sebagai contoh supervised NLP.
3. Representasi kata menggunakan one-hot encoding.
4. Embedding layer sebagai representasi kata yang dipelajari.
5. Interpretasi bobot dan kedekatan word embeddings.
6. Pembelajaran representasi kata melalui tugas *fill in the blank*.
7. Hubungan data, arsitektur, dan fungsi loss dengan pola yang dipelajari model.
8. Analogi kata sebagai salah satu sifat menarik embedding.

**Catatan reproduksi:** notebook ini merangkum ide utama bab dan menyertakan eksperimen NumPy yang dapat dijalankan tanpa mengunduh dataset IMDB. Kode contoh yang dibuat sendiri ditandai sebagai demonstrasi, bukan salinan listing kode buku.

## 2. Natural Language Processing (NLP)

**Natural Language Processing (NLP)** adalah bidang yang mempelajari bagaimana komputer memproses bahasa manusia. Tugas NLP dapat melibatkan pemberian label pada bagian teks, klasifikasi teks, atau menghasilkan teks.

Contoh tugas:
- menentukan apakah ulasan film bernada positif atau negatif;
- menentukan kategori atau label untuk suatu teks;
- memperkirakan kata yang hilang dari sebuah kalimat;
- mempelajari representasi kata dari pola kemunculan kata di dalam data.

Jaringan saraf memetakan angka masukan menjadi angka keluaran. Karena itu, teks perlu diproses terlebih dahulu menjadi representasi numerik. Tahap ini bukan sekadar detail teknis: representasi yang dipilih memengaruhi pola apa yang dapat dipelajari model.

## 3. Supervised NLP: klasifikasi sentimen ulasan film

Bab menggunakan klasifikasi ulasan film IMDB sebagai contoh. Tujuannya adalah memprediksi apakah ulasan bersifat **positif (POS)** atau **negatif (NEG)**.

Dalam supervised learning, setiap contoh pelatihan memiliki:
- **input:** representasi teks ulasan;
- **target/label:** kategori sentimen yang diketahui.

Model belajar dengan mencari hubungan antara pola kata pada ulasan dan labelnya. Misalnya, kata seperti *excellent* mungkin lebih sering muncul pada ulasan positif, sedangkan *awful* mungkin lebih sering muncul pada ulasan negatif. Namun, satu kata saja tidak selalu menentukan sentimen; konteks dan kombinasi kata juga penting.

### Alur umum
1. Kumpulkan ulasan beserta label.
2. Bangun vocabulary dari kata yang digunakan.
3. Ubah teks menjadi angka.
4. Latih jaringan untuk memprediksi label.
5. Bandingkan prediksi dengan label sebenarnya menggunakan loss.
6. Evaluasi pada data yang tidak dipakai untuk melatih model.

Notebook ini memakai dataset kecil buatan untuk mendemonstrasikan alur tersebut; hasilnya tidak mewakili eksperimen IMDB dalam buku.

## 4. One-hot encoding

One-hot encoding memberi setiap kata sebuah vektor yang panjangnya sama dengan ukuran vocabulary. Vektor tersebut berisi satu nilai `1` pada posisi kata yang diwakili dan `0` pada posisi lainnya.

Jika vocabulary adalah `["buruk", "bagus", "film"]`, representasinya dapat berupa:
- `buruk` → `[1, 0, 0]`
- `bagus` → `[0, 1, 0]`
- `film` → `[0, 0, 1]`

Kelebihannya adalah representasi ini sederhana dan setiap kata dapat dibedakan secara jelas. Keterbatasannya, vektor one-hot tidak secara langsung menyatakan bahwa dua kata memiliki makna atau fungsi yang mirip. Setiap pasangan kata berbeda memiliki vektor ortogonal, walaupun kata-kata itu mungkin digunakan pada konteks serupa.

In [ ]:
import numpy as np

vocabulary = ["buruk", "bagus", "film", "membosankan", "menarik"]
word_to_index = {word: i for i, word in enumerate(vocabulary)}

def one_hot(word, vocab_size=len(vocabulary)):
    vector = np.zeros(vocab_size, dtype=float)
    vector[word_to_index[word]] = 1.0
    return vector

for word in vocabulary:
    print(f"{word:15s} -> {one_hot(word)}")

### Interpretasi

Setiap kata memiliki posisi unik pada vocabulary. Penambahan kata baru mengubah ukuran vocabulary dan panjang vektor. Untuk vocabulary besar, one-hot vectors menjadi sangat panjang dan sebagian besar elemennya bernilai nol (*sparse representation*).

One-hot encoding adalah representasi identitas kata; representasi ini sendiri belum mempelajari hubungan semantik antarkata.

## 5. Dari one-hot encoding ke embedding layer

Salah satu cara membangun embedding adalah menggunakan matriks bobot yang baris-barisnya berkorespondensi dengan kata dalam vocabulary. Memilih embedding suatu kata setara dengan mengambil baris yang sesuai dari matriks tersebut.

Misalkan:
- ukuran vocabulary adalah \(V\);
- dimensi embedding adalah \(D\);
- matriks embedding \(E\) memiliki bentuk \(V \times D\).

Untuk kata dengan indeks \(i\), embedding-nya adalah:

\[
\mathbf{e}_i = E_{i,:}
\]

Dengan one-hot vector \(\mathbf{x}_i\), operasi yang setara dapat ditulis:

\[
\mathbf{e}_i = \mathbf{x}_i^T E
\]

Perkalian one-hot dengan matriks memilih baris yang sesuai. Dalam implementasi, framework biasanya menggunakan operasi indeks langsung agar tidak perlu membuat vektor one-hot penuh setiap kali.

In [ ]:

rng = np.random.default_rng(11)
embedding_dim = 3
embedding_matrix = rng.normal(0, 0.1, size=(len(vocabulary), embedding_dim))

def get_embedding(word):
    return embedding_matrix[word_to_index[word]]

for word in ["buruk", "bagus", "film"]:
    print(word, "->", np.round(get_embedding(word), 3))

### Mengapa embedding dapat berguna?

Embedding adalah representasi padat (*dense vector*). Berbeda dari one-hot encoding, nilai embedding dapat disesuaikan selama pelatihan. Jika tugas pelatihan mendorong kata-kata dengan fungsi atau pola prediksi serupa memiliki representasi yang serupa, informasi tersebut dapat tercermin dalam bobot embedding.

Penting: embedding acak pada contoh sebelumnya **belum** memiliki makna semantik. Hubungan semantik tidak muncul hanya karena sebuah kata diubah menjadi vektor; model perlu belajar dari data dan tujuan pelatihan.

## 6. Menggunakan embedding untuk klasifikasi sentimen

Untuk demonstrasi sederhana, kita dapat merata-ratakan embedding kata-kata dalam sebuah ulasan, lalu menggunakan model linear untuk memprediksi sentimen. Pendekatan rata-rata ini mengabaikan urutan kata, sehingga bukan model lengkap untuk memahami bahasa. Namun, contoh ini menunjukkan bagaimana representasi kata dapat digunakan dalam model klasifikasi.

Kita akan menggunakan data sintetis kecil. Ini bukan dataset IMDB asli dan tidak dimaksudkan untuk menghasilkan ukuran akurasi yang dapat digeneralisasikan.

In [ ]:
reviews = [
    ("film bagus menarik", 1),
    ("film sangat bagus", 1),
    ("cerita menarik bagus", 1),
    ("film buruk membosankan", 0),
    ("cerita buruk", 0),
    ("sangat membosankan buruk", 0),
]

sent_vocab = sorted({word for review, _ in reviews for word in review.split()})
sent_index = {word: i for i, word in enumerate(sent_vocab)}
rng = np.random.default_rng(21)
E = rng.normal(0, 0.2, size=(len(sent_vocab), 4))

def encode_review(text):
    ids = [sent_index[w] for w in text.split() if w in sent_index]
    if not ids:
        return np.zeros(E.shape[1])
    return E[ids].mean(axis=0)

X = np.vstack([encode_review(text) for text, _ in reviews])
y = np.array([label for _, label in reviews], dtype=float)

print("Vocabulary:", sent_vocab)
print("Bentuk fitur:", X.shape)
print("Bentuk label:", y.shape)

Embedding dan fitur pada contoh di atas belum dilatih, sehingga model belum otomatis dapat memisahkan sentimen. Agar pelatihan terjadi, embedding dan bobot klasifikasi perlu diperbarui berdasarkan loss. Bagian berikut memperlihatkan pelatihan classifier linear pada fitur yang tersedia; karena dataset sangat kecil, hasilnya hanya ilustrasi mekanisme optimisasi.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1.0 / (1.0 + np.exp(-z))

w = np.zeros(X.shape[1])
b = 0.0
lr = 0.5

for epoch in range(300):
    p = sigmoid(X @ w + b)
    dw = X.T @ (p - y) / len(y)
    db = np.mean(p - y)
    w -= lr * dw
    b -= lr * db

pred = (sigmoid(X @ w + b) >= 0.5).astype(int)
print("Prediksi:", pred)
print("Target  :", y.astype(int))
print("Akurasi pada data demonstrasi:", round(np.mean(pred == y), 3))

## 7. Menginterpretasikan embedding

Dalam model yang dilatih, embedding kata dapat dianalisis dengan membandingkan vektornya. Salah satu ukuran sederhana adalah **cosine similarity**:

\[
\operatorname{cos}(\mathbf{a},\mathbf{b}) =
\frac{\mathbf{a}\cdot\mathbf{b}}
{\|\mathbf{a}\|\,\|\mathbf{b}\|}
\]

Nilai yang mendekati 1 menunjukkan arah vektor yang mirip; nilai mendekati 0 menunjukkan arah yang kurang berkaitan, dan nilai negatif menunjukkan arah yang berlawanan. Interpretasi semantik tetap bergantung pada data, tugas, dan proses pelatihan.

In [ ]:
def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


pairs = [("bagus", "menarik"), ("bagus", "buruk")]
for a, b_word in pairs:
    sim = cosine_similarity(E[sent_index[a]], E[sent_index[b_word]])
    print(f"cosine({a}, {b_word}) = {sim:.3f}")

**Batas interpretasi:** karena embedding dalam contoh ini diinisialisasi secara acak dan tidak diperbarui bersama classifier, kemiripan yang dihasilkan tidak boleh dianggap sebagai temuan semantik. Pada model sebenarnya, embedding perlu dilatih dengan objective yang relevan sebelum dianalisis.

## 8. Arsitektur dan jenis informasi yang dipelajari

Arsitektur jaringan memengaruhi pola yang dapat dipelajari. Bab membedakan gagasan mempelajari korelasi langsung antara kata dan label dari pembentukan kelompok representasi pada hidden layer.

- Lapisan sederhana dapat menangkap korelasi langsung antara kata dan label target.
- Hidden layer dapat membentuk kombinasi atau kelompok fitur yang membantu prediksi.
- Kata-kata yang memiliki daya prediksi serupa untuk tugas tertentu dapat memperoleh pola bobot yang serupa.
- Interpretasi bobot harus dikaitkan dengan tugas yang dilatih. Embedding yang dipelajari untuk klasifikasi sentimen tidak harus identik dengan embedding yang dipelajari untuk memprediksi kata yang hilang.

Dengan demikian, representasi kata bukan definisi universal tentang arti kata; representasi itu dipengaruhi oleh data dan objective pelatihan.

## 9. *Fill in the blank*: mempelajari representasi dari konteks

Selain memprediksi sentimen, jaringan dapat dilatih untuk memprediksi kata yang hilang dalam suatu konteks. Misalnya, model diberi sebagian kalimat dan harus memperkirakan kata yang cocok mengisi bagian kosong.

Tugas ini mengubah target pelatihan:
- Pada klasifikasi sentimen, targetnya adalah label POS atau NEG.
- Pada *fill in the blank*, targetnya adalah kata yang benar dari vocabulary.

Karena targetnya berbeda, jaringan didorong untuk mempelajari pola yang berbeda pula. Untuk memperkirakan kata yang hilang, model perlu memanfaatkan hubungan antara kata-kata di sekitar posisi kosong, bukan hanya korelasi kata dengan sentimen.

In [ ]:

sentences = [
    ["film", "ini", "sangat", "bagus"],
    ["cerita", "ini", "sangat", "menarik"],
    ["film", "itu", "sangat", "buruk"],
]

examples = []
for tokens in sentences:
    for i, target in enumerate(tokens):
        context = tokens[:i] + tokens[i+1:]
        examples.append((context, target))

for context, target in examples[:8]:
    print(f"Konteks: {context} -> target: {target}")

Contoh di atas hanya memperlihatkan bagaimana data konteks-target dapat dibentuk. Model *fill in the blank* yang sebenarnya memerlukan representasi numerik, arsitektur prediksi, fungsi loss, dan pelatihan. Jangan menyimpulkan bahwa kode tersebut sudah melatih model bahasa.

## 10. Fungsi loss menentukan apa yang dipelajari

Fungsi loss atau objective function mengukur seberapa jauh prediksi model dari target yang diharapkan. Selama pelatihan, optimizer menggunakan gradien loss untuk memperbarui bobot.

Secara konseptual:
1. model menerima input;
2. model menghasilkan prediksi;
3. prediksi dibandingkan dengan target melalui loss;
4. gradien loss dihitung;
5. bobot diperbarui untuk mengurangi loss.

Mengganti target atau loss dapat mengubah pola yang dipelajari meskipun dataset atau sebagian arsitektur sama. Loss tidak berdiri sendiri: ia berinteraksi dengan data, arsitektur, optimizer, regularisasi, dan cara preprocessing.

## 11. Analogi kata

Word embeddings kadang menunjukkan pola relasi yang dapat diekspresikan melalui operasi vektor. Salah satu contoh populer dalam literatur adalah relasi analogi seperti pasangan jenis kelamin atau hubungan ibu kota-negara.

Namun, pola analogi tidak dijamin muncul pada setiap embedding. Hasil bergantung pada ukuran dan kualitas korpus, metode pelatihan, dimensi embedding, serta bias dalam data. Operasi vektor yang tampak berhasil juga tidak membuktikan bahwa model memahami makna seperti manusia.

## 12. Kesimpulan Bab

Bab ini memperlihatkan bahwa jaringan saraf dapat mempelajari representasi kata yang berguna untuk tugas NLP.

Poin utama:
1. Teks harus dikonversi menjadi representasi numerik sebelum diproses jaringan.
2. One-hot encoding mewakili identitas kata, tetapi tidak secara langsung menyatakan kemiripan semantik.
3. Embedding layer menyediakan representasi padat yang bobotnya dapat dipelajari.
4. Representasi yang terbentuk bergantung pada tugas, data, arsitektur, dan fungsi loss.
5. Klasifikasi sentimen dan *fill in the blank* adalah objective berbeda, sehingga dapat mendorong jaringan mempelajari informasi berbeda.
6. Kemiripan vektor dan analogi kata adalah alat analisis yang berguna, tetapi hasilnya harus ditafsirkan dengan hati-hati.

## Refleksi
- Mengapa one-hot encoding tidak secara langsung menunjukkan bahwa dua kata memiliki makna serupa?
- Mengapa embedding yang dilatih untuk klasifikasi sentimen bisa berbeda dari embedding yang dilatih untuk memprediksi kata yang hilang?
- Apa yang perlu diperiksa sebelum menyimpulkan bahwa word embeddings telah mempelajari hubungan semantik?